<a href="https://colab.research.google.com/github/Yogapmana/Yogapmana/blob/main/Costomer_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from faker import Faker
import numpy as np

# Inisialisasi Faker untuk data Indonesia
fake = Faker('id_ID')

# Muat dataset Anda
# Pastikan Anda sudah mengekstrak 'bank-additional.zip'
try:
    df = pd.read_csv('bank-additional-full.csv', sep=';')
except FileNotFoundError:
    print("Pastikan file 'bank-additional-full.csv' ada di folder yang sama.")
    exit()

print(f"Data asli dimuat, jumlah baris: {len(df)}")
print("Contoh data asli:")
print(df.head())

# Buat data palsu
jumlah_baris = len(df)
nama_palsu = [fake.name() for _ in range(jumlah_baris)]
telp_palsu = [fake.phone_number() for _ in range(jumlah_baris)]

# Buat DataFrame baru untuk data palsu
df_palsu = pd.DataFrame({
    'nama': nama_palsu,
    'no_telp': telp_palsu
})

# Tambahkan ID unik untuk referensi (bukan untuk model)
df['customer_id'] = np.arange(len(df))
df_palsu['customer_id'] = np.arange(len(df))

# df_final adalah data lengkap dengan info nasabah
df_final = pd.merge(df, df_palsu, on='customer_id')

print("\nData setelah digabung dengan info palsu:")
print(df_final[['customer_id', 'nama', 'no_telp', 'age', 'job', 'y']].head())

Data asli dimuat, jumlah baris: 41188
Contoh data asli:
   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   
3   may         mon  ...         1    999         0  nonexistent          1.1   
4   may         mon  ...         1    999         0  nonexistent          1.1   



In [ ]:
# Pisahkan fitur (X) dan target (y)
target_column = 'y'
y = df_final[target_column].map({'yes': 1, 'no': 0})

# SECARA EKSPLISIT HAPUS KOLOM YANG TIDAK DIPERLUKAN UNTUK TRAINING
# 'y' adalah target
# 'duration' adalah bocoran data (data leakage)
# 'customer_id', 'nama', 'no_telp' adalah identifier, bukan fitur prediktif
kolom_yang_dibuang = [target_column, 'duration', 'customer_id', 'nama', 'no_telp']

X = df_final.drop(columns=kolom_yang_dibuang)

print(f"\nFitur yang digunakan untuk melatih model ({len(X.columns)} fitur):")
print(X.columns.tolist())


Fitur yang digunakan untuk melatih model (19 fitur):
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Pisahkan mana kolom numerik dan mana kolom kategorikal
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print("\nFitur Numerik:", numeric_features)
print("\nFitur Kategorikal:", categorical_features)

# 2. Buat pipeline preprocessing
# Pipeline untuk data numerik: scaling
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Pipeline untuk data kategorikal: one-hot encoding
categorical_transformer = Pipeline(steps=[
    # handle_unknown='ignore' akan menangani nilai baru di data tes
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 3. Gabungkan kedua pipeline dengan ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 4. Pisahkan data menjadi data training dan data testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# stratify=y penting untuk dataset yang tidak seimbang (imbalanced) seperti ini


Fitur Numerik: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Fitur Kategorikal: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Buat pipeline lengkap: Preprocessing -> Training Model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42,
                                          n_estimators=100,
                                          class_weight='balanced')) # class_weight 'balanced' membantu data imbalanced
])

# Latih model!
print("\nMulai melatih model (menggunakan RandomForest)...")
model_pipeline.fit(X_train, y_train)
print("Model selesai dilatih.")

# Evaluasi model di data testing
y_pred = model_pipeline.predict(X_test)
y_pred_proba = model_pipeline.predict_proba(X_test)[:, 1] # Probabilitas 'yes'

print(f"\nAkurasi: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f} (Metrik Kunci untuk Ranking)")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred))


Mulai melatih model (menggunakan RandomForest)...
Model selesai dilatih.

Akurasi: 0.8957
ROC-AUC Score: 0.7817 (Metrik Kunci untuk Ranking)

Laporan Klasifikasi:
              precision    recall  f1-score   support

           0       0.91      0.97      0.94      7310
           1       0.57      0.29      0.38       928

    accuracy                           0.90      8238
   macro avg       0.74      0.63      0.66      8238
weighted avg       0.88      0.90      0.88      8238



In [ ]:
print("\nMenghasilkan skor probabilitas untuk semua nasabah...")

# Gunakan model untuk memprediksi probabilitas pada SEMUA data (X)
# Kolom [:, 1] adalah probabilitas untuk kelas '1' ('yes')
all_probabilities = model_pipeline.predict_proba(X)[:, 1]

# Tambahkan skor ini ke DataFrame 'df_final' kita
df_final['skor_probabilitas'] = all_probabilities

# Tampilkan hasilnya
print(df_final[['customer_id', 'nama', 'no_telp', 'y', 'skor_probabilitas']].head())


Menghasilkan skor probabilitas untuk semua nasabah...
   customer_id                      nama              no_telp   y  \
0            0       Vero Hartati, M.TI.      (0877) 367 5688  no   
1            1            Indra Winarsih  +62 (0503) 344 8477  no   
2            2   Rahmi Hardiansyah, M.Pd     +62-236-498-0858  no   
3            3  Pranata Zulkarnain, S.Gz   +62 (289) 435-7567  no   
4            4               Oni Suryono   +62 (020) 560 2518  no   

   skor_probabilitas  
0               0.00  
1               0.00  
2               0.00  
3               0.00  
4               0.01  


In [ ]:
# Siapkan data untuk portal
# Kita hanya butuh nama, no. telp, dan skor
portal_data = df_final[['customer_id', 'nama', 'no_telp', 'skor_probabilitas']]

# URUTKAN (RANKING) berdasarkan skor, dari tertinggi ke terendah
portal_data_prioritas = portal_data.sort_values(by='skor_probabilitas', ascending=False)

# Reset index untuk kebersihan data
portal_data_prioritas = portal_data_prioritas.reset_index(drop=True)

# Tampilkan 10 nasabah prioritas teratas untuk tim sales
print("\n=== DAFTAR NASABAH PRIORITAS UNTUK PORTAL (TOP 10) ===")
print(portal_data_prioritas.head(10))

# Simpan ke CSV untuk di-upload ke database portal Anda
portal_data_prioritas.to_csv('nasabah_prioritas_untuk_portal.csv', index=False)

print("\nSelesai! File 'nasabah_prioritas_untuk_portal.csv' telah dibuat.")


=== DAFTAR NASABAH PRIORITAS UNTUK PORTAL (TOP 10) ===
   customer_id                          nama              no_telp  \
0        40574      T. Virman Haryanti, S.H.    +62-0335-250-8870   
1        40419                  Dodo Mustofa    +62-0973-413-2953   
2        40675                 Yani Lazuardi  +62 (0892) 996-7275   
3        40653               Puti Novi Putra           0867936483   
4        40645       drg. Ika Haryanti, S.Pd     +62-313-991-1184   
5        40416      Cut Zahra Hutasoit, M.M.   +62 (009) 842-7109   
6        40663                   Vera Wijaya   +62 (501) 961 1799   
7        39215      Drs. Bella Thamrin, M.Pd   +62 (033) 888 6455   
8        39220  Sutan Dacin Permata, S.I.Kom   +62 (013) 275-8007   
9        39640         Nadine Maryati, M.TI.    +62 (81) 175 9322   

   skor_probabilitas  
0                1.0  
1                1.0  
2                1.0  
3                1.0  
4                1.0  
5                1.0  
6                1.0  
